# Kauvar, Richman, Liu et al. (2025) on dandiset 001326

- Paper: Kauvar, Richman, Liu et al., "Conserved brain-wide emergence of emotional response from sensory experience in humans and mice", Science 388, eadt3971 (2025), doi:10.1126/science.adt3971.
- Dataset: DANDI:001326, "Brain-wide mouse electrophysiology during aversive stimuli and ketamine", version 0.250528.1957 (doi:10.48324/dandi.001326/0.250528.1957), streamed from the archive.
- Version. The paper cites the dandiset without a version. 0.250528.1957 is its only published version; the draft holds the same 13 assets (same asset ids), so the most recent published version is pinned.
- Claims file: `claims/001326/10.1126-science.adt3971.yaml`, written from the accepted author manuscript (Europe PMC, PMC12286656) with its supplementary materials, and the paper's code (analysis.py, Zenodo doi:10.5281/zenodo.14876694; brainwide-npix at commit 97cdf58).
- Outside input: the Allen CCFv3 2017 structure tree (`structure_tree_safe_2017.csv`, the file brainwide-npix loads), from github.com/cortex-lab/allenCCF at commit e5a57fe7, used only to place the dataset's region acronyms in the Allen hierarchy.

The dandiset holds the mouse Neuropixels recordings (13 sessions, 10 mice). The human data, the behavior-only mouse experiments and the eye closure of the recorded mice are in DANDI:001347; the claims on them that the data allow are tested in `reproduce_kauvar_2025_behavior.ipynb`.


| Claim | In this notebook |
|---|---|
| `fig1e-human-ketamine-late-closure` | tested in reproduce_kauvar_2025_behavior.ipynb (DANDI:001347) |
| `figs1b-human-early-closure-unchanged` | tested in reproduce_kauvar_2025_behavior.ipynb (DANDI:001347) |
| `fig1f-human-reports` | not tested (see the attempt record) |
| `fig2d-cadss-ketamine` | not tested (see the attempt record) |
| `fig2g-six-factors` | not tested (see the attempt record) |
| `fig2i-factors-1-2-ketamine` | not tested (see the attempt record) |
| `fig3b-cluster-test-networks` | not tested (see the attempt record) |
| `figs12-erp-not-blink` | not tested (see the attempt record) |
| `figs14g-cadss-correlation` | not tested (see the attempt record) |
| `fig6c-human-persistent` | not tested (see the attempt record) |
| `fig6e-human-fast-unchanged` | not tested (see the attempt record) |
| `fig6g-human-timescale` | not tested (see the attempt record) |
| `fig6i-human-phase-locking` | not tested (see the attempt record) |
| `fig1k-mouse-ketamine-late-closure` | tested in reproduce_kauvar_2025_behavior.ipynb (DANDI:001347) |
| `fig1l-dissociatives-preinfusion` | tested in reproduce_kauvar_2025_behavior.ipynb (DANDI:001347) |
| `figs2bc-mouse-early-late` | tested in reproduce_kauvar_2025_behavior.ipynb (DANDI:001347) |
| `fig1m-accumulation-persistence` | tested in reproduce_kauvar_2025_behavior.ipynb (DANDI:001347) |
| `figs2g-scaling-puff-number` | tested in reproduce_kauvar_2025_behavior.ipynb (DANDI:001347) |
| `figs2h-dose-response` | tested in reproduce_kauvar_2025_behavior.ipynb (DANDI:001347) |
| `fig1p-anesthesia-abolishes-both` | tested in reproduce_kauvar_2025_behavior.ipynb (DANDI:001347) |
| `figs5-other-drugs` | tested in reproduce_kauvar_2025_behavior.ipynb (DANDI:001347) |
| `figs6-vpm-muscimol` | tested in reproduce_kauvar_2025_behavior.ipynb (DANDI:001347) |
| `figs4-reward-licking` | not tested (see the attempt record) |
| `fig5hi-behavior-model-doses` | tested in reproduce_kauvar_2025_behavior.ipynb (DANDI:001347) |
| `fig4-recordings-scale` | tested here |
| `fig4b-recorded-eye-closure-ketamine` | tested in reproduce_kauvar_2025_behavior.ipynb (DANDI:001347) |
| `fig5b-recorded-accumulation` | tested in reproduce_kauvar_2025_behavior.ipynb (DANDI:001347) |
| `fig4de-response-clusters` | tested here |
| `figs31b-cluster-overrepresentation` | tested here |
| `fig4g-ketamine-alters-slow-clusters` | tested here |
| `fig4i-rise-times-unchanged` | tested here |
| `fig4h-decay-shortened` | tested here |
| `fig4h-decay-anatomical-progression` | tested here |
| `fig5d-emotion-dimension-accumulates` | tested here |
| `fig5e-late-reduced` | tested here |
| `fig5e-rapid-unchanged` | tested here |
| `fig5g-tau-persist-model` | tested here |
| `figs33-alternative-models` | tested here |
| `figs32c-regional-emotion-dimension` | tested here |
| `fig6b-persistent-dimension-distributed` | tested here |
| `fig6d-mouse-persistent` | tested here |
| `fig6f-mouse-fast-unchanged` | tested here |
| `fig6h-mouse-timescale` | tested here |
| `fig6j-mouse-coupling` | tested here |
| `figs34c-tone-no-eye-closure` | tested in reproduce_kauvar_2025_behavior.ipynb (DANDI:001347) |
| `figs34d-tone-fast-unchanged` | tested here |
| `figs34e-tone-persistent` | tested here |
| `figs34f-tone-shorter-than-puff` | tested here |
| `figs34g-tone-timescale` | tested here |
| `figs34h-tone-coupling` | tested here |
| `figs30-correlation-structure` | tested here |
| `figs29-rsp-oscillations` | tested here |
| `link-001347-condition-labels` | not tested (see the attempt record) |
| `link-001347-recorded-sessions` | not tested (see the attempt record) |

## Loading

Each file is streamed and reduced as it loads (`kauvar.process_session`): rates in 10 ms bins with a causal 50 ms moving average, z-scored per unit, the puff-modulated units of analysis.py `selective`, and every per-session quantity the claims need. Nothing is written to disk.

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import scipy
import scipy.cluster
import scipy.spatial
import scipy.stats
import kauvar as K

warnings.filterwarnings("ignore", category=RuntimeWarning)
print(f"dandiset {K.DANDISET}, version {K.VERSION}")
urls = K.asset_urls()
paths = K.allen_tree()
res, meta = {}, {}
t0 = time.time()
for p, u in urls:
    s = K.load_session(p, u)
    meta[s.name] = dict(subject=s.subject, units=len(s.spikes), unsorted_units=s.n_unsorted_units, spikes=s.n_spikes_total,
                        n_spikes_column=int(s.n_spikes.sum()), snr_min=s.snr.min(), n_spikes_min=int(s.n_spikes.min()),
                        quality=sorted(set(s.quality)), duration_s=s.maxt, stim=s.stim.stimulus.value_counts().to_dict(),
                        stim_dur=s.stim.groupby("stimulus").apply(lambda g: (g.stop_time - g.start_time).agg(["min", "max"]).round(4).tolist()).to_dict(),
                        puff_isi=np.round(np.percentile(np.diff(s.onsets("puff").reshape(-1, 8), axis=1), [0, 100]), 3).tolist(),
                        series_gap=np.round(np.percentile(np.diff(s.onsets("puff").reshape(-1, 8)[:, 0]), [0, 100]), 1).tolist(),
                        **s.meta)
    res[s.name] = K.process_session(s, paths)
    del s
names = list(res)
print(f"{len(names)} sessions loaded and reduced in {time.time() - t0:.0f} s")

## Data check

What the files hold, checked against the paper's description (Methods, "Neuropixels recording and processing"; claims file, opening comment). The clock check compares the mean z-scored population response 0 to 200 ms after each stimulus onset of the table (minus the 1 s before) with the same measure at onsets shifted by 1.5 s, between stimuli: a response at the table's onsets and none at the shifted ones shows that spike times and stimulus table share a clock.


In [ ]:
m = pd.DataFrame(meta).T
print("Units table: every unit is labelled", sorted({q for v in m.quality for q in v}), "; smallest SNR",
      round(m.snr_min.min(), 2), "(paper: SNR above 2); smallest spike count", m.n_spikes_min.min(), "(paper: more than 500)")
print("n_spikes column equals the stored spike times in every file:", bool((m.spikes == m.n_spikes_column).all()))
print("units whose spike times are not in ascending order:", int(m.unsorted_units.sum()), "of", int(m.units.sum()))
print(m[["subject", "units", "spikes", "duration_s", "devices", "age", "sex"]].to_string())
print("\nStimulus presentations per session (count) and duration range (s):")
print(pd.DataFrame({k: v for k, v in m.stim.items()}).T.fillna(0).astype(int).to_string())
print(pd.Series(m.stim_dur).to_string())
print("\nPuff onset intervals within a series (s, min and max over sessions):", [min(v[0] for v in m.puff_isi), max(v[1] for v in m.puff_isi)])
print("Intervals between series starts (s, min and max):", [min(v[0] for v in m.series_gap), max(v[1] for v in m.series_gap)])
rows = []
for k in names:
    o = res[k]
    rows.append({"session": k, "subject": o["subject"], "puff series": o["n_series"], "series of 8": set(o["series_sizes"]) == {8},
                 "code ketamine onset (s)": round(o["ket_start_code"], 1), "infusion in file (s)": round(o["ket_infusion_nwb"], 1),
                 "tone series": len(o.get("tone_series_sizes", [])), "regions (finest, no NA)": len(set(o["regions_all"]) - {"NA"}),
                 "units": o["n_units"], "puff-modulated": o["n_mod"],
                 "clock puff true": round(o["clock_puff"]["true"], 3), "clock puff shifted": round(o["clock_puff"]["shifted 1.5 s"], 3),
                 "clock tone true": round(o["clock_tone"]["true"], 3) if "clock_tone" in o else np.nan,
                 "clock tone shifted": round(o["clock_tone"]["shifted 1.5 s"], 3) if "clock_tone" in o else np.nan})
sess = pd.DataFrame(rows).set_index("session")
print(sess.to_string())
print(f"\nsessions {len(names)}, mice {sess.subject.nunique()}, units {sess.units.sum()}, puff-modulated {sess['puff-modulated'].sum()}")
print("every session has at least 40 puff series:", bool((sess["puff series"] >= 40).all()),
      "; the infusion in the file falls after series 20 in every session:", bool((sess["infusion in file (s)"] > sess["code ketamine onset (s)"] - 30).all()))
print("rates: z-scored per unit; z values of the stored windows range",
      round(min(res[k]["pre_avg"].min() for k in names), 2), "to", round(max(res[k]["pre_avg"].max() for k in names), 2), "(Pre averages)")

## fig4-recordings-scale

> Many thousands of single units were recorded across 13 sessions from 10 mice, with dozens of regions recorded simultaneously.

(Figure 4C; fig. S29A and S29B; table S2; table S3, Fig. 4E)

Counts from the files. The units in the files are the curated units (all labelled good, SNR at least 2, more than 500 spikes, see the data check); "puff-modulated" is analysis.py `selective` with `has_min_spikes` on the summed Hz rate (claims file, opening comment). Regions are the finest Allen acronyms in `brain_region`, without the unassigned label `NA`.


In [ ]:
n_sessions, n_mice = len(names), sess.subject.nunique()
n_mod = int(sess["puff-modulated"].sum())
print(f"sessions: {n_sessions} (expected 13); mice: {n_mice} (expected 10)")
print(f"units after the quality criteria (all units in the files): {sess.units.sum()} (range per session {sess.units.min()}-{sess.units.max()})")
print(f"puff-modulated units: {n_mod} (expected 6969, tolerance 697); with the spike-count reading of the 250 minimum: "
      f"{sum(res[k]['sel_count_reading'] for k in names)}")
print("per test (units passing, summed over sessions):", pd.DataFrame({k: res[k]["n_test"] for k in names}).sum(1).to_dict())
reg = sess["regions (finest, no NA)"]
print(f"regions per session at the finest Allen level: median {reg.median():.0f}, range {reg.min()}-{reg.max()}; "
      f"sessions with at least 24: {(reg >= 24).sum()} of {len(reg)}")

**Result: reproduced.** The files hold 13 sessions from 10 mice and 9554 curated units; 6698 are puff-modulated, within 697 of the 6969 of table S3. The median session has units in 32 regions at the finest Allen level (range 11 to 37), above two dozen.

## fig4de-response-clusters

> Unsupervised clustering of peri-puff responses of puff-modulated units from all sessions gave twelve clusters spread over many regions, some responding rapidly and briefly and others persistently after rapid (clusters 0 and 1) or delayed (cluster 2) onset.

(Figure 4D and 4E; fig. S31A; table S3, Fig. 4E)

Method from the criterion: per unit the Pre average (160 puffs) of the z-scored rate from -1 to 2 s, minus its mean over -1 to 0 s; scanpy PCA (arpack), neighbors (30, 20 PCs), UMAP, Leiden with default resolution, `np.random.seed(42)` and scanpy's default seeds, as get_cell_clusters. Cluster means and the decay to 25% of the post-puff peak follow analysis.py (`get_cell_clusters` cluster averages; `find_decay` in plot_cluster_distribution, which takes absolute values and gives 2 s to a cluster whose peak has |z| below 1). "Persistent" is read as the mean absolute cluster trace over 1 to 2 s above 25% of the absolute peak (criterion); the minimum over 1 to 2 s is also printed.


In [ ]:
import scanpy as sc

pre_avg = np.vstack([res[k]["pre_avg"] for k in names])
ket_avg = np.vstack([res[k]["ket_avg"] for k in names])
mod_regions = np.hstack([res[k]["regions"] for k in names])
mod_session = np.hstack([[k] * res[k]["n_mod"] for k in names])
X = np.nan_to_num(pre_avg - pre_avg[:, :100].mean(1, keepdims=True)).astype(np.float32)
np.random.seed(42)
ann = sc.AnnData(X)
sc.tl.pca(ann, svd_solver="arpack")
sc.pp.neighbors(ann, n_neighbors=30, n_pcs=20)
sc.tl.umap(ann)
sc.tl.leiden(ann, flavor="leidenalg")
labels = ann.obs["leiden"].astype(int).to_numpy()
nclus = labels.max() + 1
sizes = np.bincount(labels)
expected_sizes = [1070, 832, 803, 663, 639, 597, 547, 508, 473, 344, 317, 176]
print(f"puff-modulated units clustered: {len(labels)}; clusters: {nclus} (expected 12, tolerance 1)")
srt = np.sort(sizes)[::-1]
rows = []
for i, e in enumerate(expected_sizes):
    got = srt[i] if i < len(srt) else 0
    rows.append({"rank": i, "size": got, "expected": e, "tolerance": round(0.1 * e), "within": abs(got - e) <= round(0.1 * e)})
print(pd.DataFrame(rows).to_string(index=False))

def cluster_means(avg, labels):
    """analysis.py get_cell_clusters: per cell z-scored over -1..2 s, averaged, minus the mean of -1..0 s."""
    out = []
    for c in range(labels.max() + 1):
        zz = np.nanmean(scipy.stats.zscore(avg[labels == c], axis=1), 0)
        out.append(zz - zz[:100].mean())
    return np.array(out)

def peak_decay(trace, frac=0.25):
    """analysis.py find_decay: |trace| normalized by its largest |value| after onset; the peak is its argmax and
    the decay the time from onset until it falls below frac after the peak; a peak whose z-score (over the
    trace) is below 1 in absolute value gets decay 2 s, as the code does. Returns peak value (signed),
    peak time, decay."""
    zt = scipy.stats.zscore(trace)
    a = np.abs(trace / np.nanmax(np.abs(trace[100:])))
    pk = int(np.argmax(a[100:]))
    below = np.where(a[100:][pk:] < frac)[0]
    decay = (pk + below[0]) * K.BIN if len(below) else np.nan
    if abs(zt[100:][pk]) < 1:
        decay = 2.0
    return trace[100 + pk], pk * K.BIN, decay

cm = cluster_means(X, labels)
# hierarchical reordering as get_cell_clusters (moving average of 20 bins, correlation, cosine, average linkage)
cp = np.array([K.moving_avg(c, 20) for c in cm])
link = scipy.cluster.hierarchy.linkage(scipy.spatial.distance.pdist(np.nan_to_num(np.corrcoef(cp)), metric="cosine"), method="average")
reordering = scipy.cluster.hierarchy.leaves_list(link)
disp = {c: i for i, c in enumerate(reordering)}       # figure number of each size-ranked cluster
shape = []
for c in range(nclus):
    pk, tpk, dec = peak_decay(cm[c])
    late = np.abs(cm[c][200:300]).mean()
    shape.append({"cluster (size rank)": c, "figure number": disp[c], "cells": sizes[c], "peak": round(pk, 2),
                  "peak time s": tpk, "decay to 25% s": dec, "mean |1-2 s| / |peak|": round(late / abs(pk), 2), "min |1-2 s| / |peak|": round(np.abs(cm[c][200:300]).min() / abs(pk), 2),
                  "rapid and brief": (dec <= 0.25) if not np.isnan(dec) else False,
                  "persistent": late > 0.25 * abs(pk), "sign": "up" if pk > 0 else "down"})
shape = pd.DataFrame(shape)
print(shape.to_string(index=False))
rb = shape["rapid and brief"].sum()
pers = shape[shape.persistent]
print(f"rapid and brief clusters: {rb}; persistent clusters: {len(pers)}, of which peak within 250 ms: "
      f"{(pers['peak time s'] < 0.25).sum()}, later: {(pers['peak time s'] >= 0.25).sum()}")
for numbering in ("cluster (size rank)", "figure number"):
    sub = shape.set_index(numbering).loc[[0, 1, 2]]
    print(f"clusters 0, 1, 2 by {numbering}: persistent {list(sub.persistent)}, peak times {list(sub['peak time s'])}")
print("cells per cluster by session:")
print(pd.crosstab(mod_session, labels).to_string())

**Result: partially reproduced.** Leiden gives 11 clusters, within one of 12. Seven of the first eleven sizes in descending order are within tolerance; the largest (941 against 1070), the 9th to 11th (526, 392, 187) and the missing 12th are not. By the criterion's measures two clusters are rapid and brief (decay 0.10 and 0.24 s) and only one is persistent (mean |1 to 2 s| at 0.34 of the peak, peak at 0.28 s), so the shape condition (three persistent clusters, two with early peaks) fails. Four clusters decay after more than 1 s (1.03 to 1.21 s), two of them with peaks at 0.51 and 0.53 s, but their mean over 1 to 2 s is 0.14 to 0.16 of the peak, below the 25% the criterion asks.

## figs31b-cluster-overrepresentation

> Clusters overrepresented in the periaqueductal gray and VPM thalamus had fast, sharp responses, while clusters overrepresented in frontal cortex and limbic regions responded more slowly and persistently.

(Figure 4G; fig. S31A and S31B; table S3, fig. S31 B)

As analysis.py get_cluster_null_distribution: regions (finest acronym, `NA` left out) with more than 30 clustered cells; the null is 20,000 draws of uniform labels (fraction in label 0), two-sided against 1/n; Benjamini-Hochberg across regions per cluster; overrepresented means significant with a fraction above 1/n. Clusters are ranked by find_decay; with 11 clusters the faster half is the 5 fastest. Frontal and limbic regions are those under ORB, ACA, PL, ILA, MOs or AI in the Allen tree.


In [ ]:
rng = np.random.default_rng(2024)
dec_c = np.array([peak_decay(cm[c])[2] for c in range(nclus)])
speed_rank = np.argsort(np.nan_to_num(dec_c, nan=99), kind="stable")    # fastest first
fast_half = set(speed_rank[: nclus // 2])
print("cluster decay times (s), fastest first:", [(int(c), dec_c[c]) for c in speed_rank])
print("faster half:", sorted(int(c) for c in fast_half))
valid = mod_regions != "NA"
ureg, ucnt = np.unique(mod_regions[valid], return_counts=True)
big = ureg[ucnt > 30]
pvals, fracs = {}, {}
for r in big:
    ncell = (mod_regions == r).sum()
    null = (rng.integers(nclus, size=(20000, ncell)) == 0).mean(1)     # analysis.py: label 0 of uniform draws
    obs = np.bincount(labels[mod_regions == r], minlength=nclus) / ncell
    pvals[r] = np.array([np.mean(np.abs(null - 1 / nclus) >= abs(o - 1 / nclus)) for o in obs])
    fracs[r] = obs
P = pd.DataFrame(pvals).T
F = pd.DataFrame(fracs).T
Q = P.apply(lambda col: pd.Series(scipy.stats.false_discovery_control(np.maximum(col.values, 1 / 20000)), index=col.index))
FRONTAL = ["ORB", "ACA", "PL", "ILA", "MOs", "AI"]
def is_frontal(r):
    return any(K.under(r, f, paths) for f in FRONTAL)
print(f"regions with more than 30 clustered cells: {len(big)}")
for thr in (0.05, 0.01):
    over = (Q < thr) & (F > 1 / nclus)
    print(f"\nFDR threshold {thr}: significantly overrepresented (region: clusters)")
    for r in big:
        cl = list(np.where(over.loc[r])[0])
        if cl:
            print(f"  {r}: {cl}" + ("  frontal/limbic" if is_frontal(r) else ""))
    pag = [c for c in np.where(over.loc["PAG"])[0] if c in fast_half] if "PAG" in big else []
    vpm = [c for c in np.where(over.loc["VPM"])[0] if c in fast_half] if "VPM" in big else []
    fr = [(r, c) for r in big if is_frontal(r) for c in np.where(over.loc[r])[0] if c not in fast_half]
    print(f"  PAG in a faster-half cluster: {pag}; VPM in a faster-half cluster: {vpm}; frontal/limbic in a slower-half cluster: {fr}")

**Result: reproduced.** Under both FDR thresholds (0.05 and 0.01), PAG is overrepresented in clusters 4 and 6 and VPM in cluster 2, all in the faster half by decay time, and frontal or limbic regions (for example ACAv5, AId5, ORBl5 and PL2/3 in cluster 0, ORBvl in clusters 1 and 9) are overrepresented in slower-half clusters. VPM is also overrepresented in cluster 3, which is in the slower half.

## fig4g-ketamine-alters-slow-clusters

> Ketamine substantially changed the responses of clusters with slow, persistent preinfusion dynamics but left fast clusters little changed.

(Figure 4G; fig. S31C; table S3, Fig 4G)

Readings from the criterion. Ketamine is series 23 to 32 (code). The Pre decay time of each cluster is find_decay on the same Pre cluster trace.


In [ ]:
def cluster_trace(avg, mask):
    """Table S3, Fig. 4G: cells of the cluster pooled, baseline-subtracted (1 s before the puff), z-scored over -1..2 s."""
    m = np.nanmean(avg[mask] - avg[mask][:, :100].mean(1, keepdims=True), 0)
    return scipy.stats.zscore(m)

rows = []
for c in range(nclus):
    tp = cluster_trace(pre_avg, labels == c)
    tk = cluster_trace(ket_avg, labels == c)
    rows.append({"cluster": c, "cells": sizes[c], "pre decay s": peak_decay(tp)[2],
                 "cosine distance": scipy.spatial.distance.cosine(tp, tk),
                 "|diff| 1-2 s": abs(tp[200:300].mean() - tk[200:300].mean())})
g = pd.DataFrame(rows)
print(g.round(3).to_string(index=False))
for meas in ("cosine distance", "|diff| 1-2 s"):
    r = scipy.stats.spearmanr(g["pre decay s"], g[meas])
    print(f"{meas}: Spearman rho = {r.statistic:.3f}, p = {r.pvalue:.4f} (n = {nclus} clusters)")

**Result: not reproduced.** Across the 11 clusters, the change from Pre to Ketamine does not correlate with the Pre decay time under either reading: Spearman rho = 0.127 (p = 0.71) for the cosine distance and rho = 0.209 (p = 0.54) for the difference over 1 to 2 s. The largest changes are in clusters 0 and 5 (cosine distance 1.43 and 0.70, Pre decay 0.69 and 0.83 s), but the fast cluster 10 also changes (0.49, decay 0.19 s) while the slow cluster 9 changes little (0.16, decay 0.95 s).

## fig4i-rise-times-unchanged

> Most regions responded to the puff with fast rise times, and ketamine did not change regional rise times.

(Figure 4I; fig. S31D to S31F; table S3, Fig. 4H, I)

As analysis.py region_time_course: regions merged across layers (reduce_region), per region and cluster the first two puffs of each series averaged over cells (40 Pre replicates, 20 Ketamine), baseline-subtracted, d' against the 1 s before the puff, rise and decay at 25% of the normalized d' peak, peaks after 750 ms dropped, cluster values combined with the region's cell fractions (clusters with one cell weighted 0), 1000 bootstraps over puff replicates, the catch-all labels NA, STR, MB and HY dropped. The code skips cluster 11; this clustering has 11 clusters (0 to 10), so none is skipped. Bootstrap seed 0 (a standard choice). The d' threshold (1.3 Methods and code, 2 table S3), the region rule (more than 30, code; at least 30, Methods) and the Ketamine series (23 to 32, code; 22 to 32, Methods-literal) are all shown.


In [ ]:
first2_pre = np.concatenate([res[k]["first2_pre"] for k in names])      # (cells, 20, 2, 300)
first2_ket = np.concatenate([res[k]["first2_ket"] for k in names])      # (cells, 10, 2, 300), series 23-32
first2_ket_lit = np.concatenate([res[k]["first2_ket_lit"] for k in names])  # series 22-32
red_regions = np.array([K.reduce_region(r) for r in mod_regions])

def dprime_abs(p):
    """region_time_course calculate_dprime on (..., trials, 300): |mean - baseline mean| / pooled sd."""
    bm = p[..., :100].mean((-1, -2))
    bs = np.nanstd(p[..., :100].reshape(p.shape[:-2] + (-1,)), -1)
    rm = p.mean(-2)
    rs = np.nanstd(p, -2)
    return np.abs(rm - bm[..., None]) / np.sqrt(0.5 * (bs[..., None] ** 2 + rs ** 2))

def rise_decay(p, thr=0.25):
    """region_time_course _find_rise_decay, vectorized over leading axes of p (..., trials, 300)."""
    d = dprime_abs(p)
    d = d - d[..., :100].mean(-1, keepdims=True)
    d = d / d.max(-1, keepdims=True)
    post = d[..., 100:]
    pk = post.argmax(-1)
    above = post > thr
    rise = np.where(above.any(-1), above.argmax(-1) * K.BIN, np.nan)
    idx = np.arange(post.shape[-1])
    after = (post < thr) & (idx >= pk[..., None])
    decay = np.where(after.any(-1), after.argmax(-1) * K.BIN, np.nan)
    bad = pk > 75
    rise[bad] = np.nan
    decay[bad] = np.nan
    return rise, decay

def region_times(first2_p, first2_k, min_cells=30, strict=True, dthr=1.3, nreps=1000, seed=0):
    rng = np.random.default_rng(seed)
    out = {}
    reg_u, reg_n = np.unique(red_regions[red_regions != "NA"], return_counts=True)
    for r, nr in zip(reg_u, reg_n):
        if (nr <= min_cells) if strict else (nr < min_cells):
            continue
        f = red_regions == r
        allp = first2_p[f].mean(0).reshape(-1, 300)
        allp = allp - allp[:, :100].mean(1, keepdims=True)
        peak = np.max(dprime_abs(allp[None])[0][100:])
        if peak < dthr:
            continue
        w = np.array([(f & (labels == c)).sum() for c in range(nclus)], float)
        w[w <= 1] = 0
        w /= w.sum()
        res_r = {"n": int(nr), "peak_dprime": peak}
        for cond, arr in (("pre", first2_p), ("ket", first2_k)):
            data = []
            for c in range(nclus):
                m = f & (labels == c)
                if w[c] == 0:
                    data.append(np.zeros((arr.shape[1] * 2, 300)))
                    continue
                d = np.nanmean(arr[m], 0).reshape(-1, 300)
                data.append(d - d[:, :100].mean(1, keepdims=True))
            data = np.stack(data)                                   # (clusters, trials, 300)
            nt = data.shape[1]
            ix = rng.integers(0, nt, size=(nreps, nt))
            ri = np.empty((nclus, nreps)); de = np.empty((nclus, nreps))
            for c in range(nclus):                                  # (reps, trials, 300) per cluster
                ri[c], de[c] = rise_decay(data[c][ix].astype(np.float32))
            res_r[cond + "_rise"] = np.nansum(w[:, None] * np.where(w[:, None] > 0, ri, 0), 0)
            res_r[cond + "_decay"] = np.nansum(w[:, None] * np.where(w[:, None] > 0, de, 0), 0)
        out[r] = res_r
    for catch in ("NA", "STR", "MB", "HY"):          # region_time_course drops these catch-all labels
        out.pop(catch, None)
    return out

# One pass per Ketamine reading at the widest settings (>= 30 cells, peak d' >= 1.3); the code's rule
# (> 30 cells) and the d' threshold of 2 (table S3) are subsets of it, applied below.
rt_all = {kname: region_times(first2_pre, karr, min_cells=30, strict=False, dthr=1.3)
          for kname, karr in (("ket 23-32", first2_ket), ("ket 22-32", first2_ket_lit))}
rt = {}
for dthr in (1.3, 2.0):
    for label, keep in (("code >30", lambda o: o["n"] > 30), ("Methods >=30", lambda o: o["n"] >= 30)):
        for kname, out in rt_all.items():
            rt[(dthr, label, kname)] = {r: o for r, o in out.items() if keep(o) and o["peak_dprime"] >= dthr}
rows = []
for key, out in rt.items():
    med_change = np.array([np.nanmedian(o["ket_rise"] - o["pre_rise"]) for o in out.values()])
    pre_rise = np.array([np.nanmedian(o["pre_rise"]) for o in out.values()])
    rows.append({"d' threshold": key[0], "region rule": key[1], "ketamine series": key[2], "regions": len(out),
                 "median over regions of median rise change (s)": np.median(med_change),
                 "regions with Pre median rise < 0.1 s": int((pre_rise < 0.1).sum()),
                 "within one bin": abs(np.median(med_change)) <= 0.01 + 1e-9})
print(pd.DataFrame(rows).to_string(index=False))
main = rt[(1.3, "code >30", "ket 23-32")]
print(pd.DataFrame({r: {"n": o["n"], "peak d'": round(o["peak_dprime"], 2), "Pre rise": np.nanmedian(o["pre_rise"]),
                        "Ket rise": np.nanmedian(o["ket_rise"]), "Pre decay": np.nanmedian(o["pre_decay"]),
                        "Ket decay": np.nanmedian(o["ket_decay"])} for r, o in main.items()}).T.sort_values("Pre decay").to_string())

**Result: reproduced.** Under every combination of d' threshold (1.3 or 2), region rule and Ketamine series, the median over regions of the bootstrap median change in rise time is -0.002 s, within one 10 ms bin of zero, and the Pre median rise is below 0.1 s in 31 of 35 regions (threshold 1.3) and 29 of 31 (threshold 2).

## fig4h-decay-shortened

> Ketamine shortened the time for regional puff responses to decay from their peak.

(Figure 4H; fig. S31G)

The bootstrapped decay times of fig4i-rise-times-unchanged.


In [ ]:
rows = []
for key, out in rt.items():
    ch = np.concatenate([o["ket_decay"] - o["pre_decay"] for o in out.values()])
    shorter = sum(np.nanmedian(o["ket_decay"]) < np.nanmedian(o["pre_decay"]) for o in out.values())
    rows.append({"d' threshold": key[0], "region rule": key[1], "ketamine series": key[2], "regions": len(out),
                 "median of pooled bootstrap changes (s)": np.nanmedian(ch),
                 "regions with Ket median decay < Pre": shorter})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** The median of the pooled bootstrapped changes in decay time is -0.16 to -0.17 s under every setting, and the Ketamine median decay is below the Pre median in 33 of 35 regions (threshold 1.3, code series) and 31 of 31 (threshold 2).

## fig4h-decay-anatomical-progression

> Before ketamine, regional decay times followed a progression from subcortical to frontal cortical regions, with long decays in frontal and limbic cortex and in VPM and MD thalamus.

(Figure 4H; Results, Brain-wide activity screen during emergence of affect in mice)

Pre decay medians of fig4i-rise-times-unchanged (code region rule, Ketamine 23 to 32). Frontal and limbic regions are those under ORB, ACA, PL, ILA, MOs, MOp or AI; midbrain those under MB, in the Allen tree.


In [ ]:
FRONTAL_LIMBIC = ["ORB", "ACA", "PL", "ILA", "MOs", "MOp", "AI"]
rows = []
for key, out in rt.items():
    if key[1] != "code >30" or key[2] != "ket 23-32":
        continue
    dec = pd.Series({r: np.nanmedian(o["pre_decay"]) for r, o in out.items()})
    fl = [r for r in dec.index if any(K.under(r, f, paths) for f in FRONTAL_LIMBIC)]
    mb = [r for r in dec.index if K.under(r, "MB", paths)]
    med_all = dec.median()
    rows.append({"d' threshold": key[0], "frontal/limbic regions": ", ".join(fl), "median decay frontal/limbic": dec[fl].median(),
                 "midbrain regions": ", ".join(mb), "median decay midbrain": dec[mb].median(), "median all": med_all,
                 "VPM": dec.get("VPM", np.nan), "MD": dec.get("MD", np.nan)})
print(pd.DataFrame(rows).T.to_string())

**Result: reproduced.** The median Pre decay of the frontal and limbic regions (0.51 s at threshold 1.3, 0.52 s at 2) exceeds that of the midbrain regions MRN, NPC, PAG and SCig (0.29 s), and VPM (0.53 s) and MD (0.57 s) both lie above the median of all regions (0.44 s).

## Session readings for Figures 5 and 6

"For cross-puff and population analyses, we excluded one animal that exhibited notable stress response (Harderian gland secretion) during the pre-ketamine epoch" (Methods, Neuropixels recording and processing). The animal is not named. Each mouse with two sessions (sub-1, sub-3, sub-5) is a reading: removing any of them leaves 11 sessions from 9 mice, as the paper reports. Only the removal of sub-5 also leaves 7 tone sessions from 7 mice, the tone counts of table S3 (fig. S34); the other two leave 8 tone sessions from 7 mice. Every session has at least 64 puff-modulated units, above the code's minimum of 20.


In [ ]:
# Session readings for the cross-puff and population analyses (Figs. 5 and 6): the paper excludes one
# unnamed animal; each mouse with two sessions is a reading (fig5d-emotion-dimension-accumulates).
multi = sorted({res[k]["subject"] for k in names if sum(res[j]["subject"] == res[k]["subject"] for j in names) > 1})
READINGS = {f"without sub-{m}": [k for k in names if res[k]["subject"] != m and res[k]["n_mod"] >= 20] for m in multi}
for r, ks in READINGS.items():
    print(f"{r}: {len(ks)} sessions from {len({res[k]['subject'] for k in ks})} mice")
TONE_SESSIONS = [k for k in names if "tone" in res[k]]
for r, ks in READINGS.items():
    t = [k for k in ks if k in TONE_SESSIONS]
    print(f"  tone sessions {r}: {len(t)} sessions from {len({res[k]['subject'] for k in t})} mice")

def bh(p):
    return scipy.stats.false_discovery_control(np.asarray(p))

def ci95(x):
    x = np.asarray(x, float)
    m = x.mean(); h = 1.96 * x.std(ddof=0) / np.sqrt(len(x))
    return m, m - h, m + h

## fig6d-mouse-persistent

> In mice, activity along the persistent (0.5 to 1 s) coding dimension 1 to 2 s after the puff fell during ketamine (0.86 to 0.02) and partly recovered afterwards (0.52).

(Figure 6D; table S3, Fig. 6D)

Criterion: weights from the Pre series means over 0.5 to 1 s against the 1 s before the puff (analysis.py plot_coding_dimension_early_late averages the eight puffs of a series before computing the weights), per-unit variance (Methods) with the pooled variance of the code as a variant; projection of the first puff of each series averaged over series (code; all eight puffs a variant), z-scored over -1 to 2 s, minus its mean over -1 to 0 s, quantified over 1 to 2 s. Epochs: code (Ketamine 23 to 32, Post 35 to 40) and Methods-literal (22 to 32, 34 to 40). Paired t-tests, Benjamini-Hochberg over the two.


In [ ]:
def zproj(trace, base="own"):
    z = scipy.stats.zscore(trace)
    return z - z[:100].mean()

rows = []
per_session = {}
for rd, ks in READINGS.items():
    for var in ("unit", "pooled"):
        for puffs in ("first", "all"):
            for ep in ("code", "Methods-literal"):
                kk = ("ket", "post") if ep == "code" else ("ket_lit", "post_lit")
                vals = {c: np.array([zproj(res[k]["cd_proj"][("persist", var, cc)][puffs])[200:300].mean() for k in ks])
                        for c, cc in (("Pre", "pre"), ("Ketamine", kk[0]), ("Post", kk[1]))}
                t1 = scipy.stats.ttest_rel(vals["Pre"], vals["Ketamine"])
                t2 = scipy.stats.ttest_rel(vals["Post"], vals["Ketamine"])
                q = bh([t1.pvalue, t2.pvalue])
                means = [vals[c].mean() for c in ("Pre", "Ketamine", "Post")]
                within = [abs(means[0] - 0.86) <= 0.1, abs(means[1] - 0.02) <= 0.25, abs(means[2] - 0.52) <= 0.41]
                rows.append({"reading": rd, "variance": var, "puffs": puffs, "epochs": ep, "Pre": means[0], "Ket": means[1], "Post": means[2],
                             "t Pre-Ket": t1.statistic, "q Pre-Ket": q[0], "t Post-Ket": t2.statistic, "q Post-Ket": q[1],
                             "means within": all(within), "tests hold": q[0] < 0.05 and q[1] < 0.05 and means[1] < min(means[0], means[2])})
                if var == "unit" and puffs == "first" and ep == "code":
                    per_session[rd] = pd.DataFrame(vals, index=ks)
f6d = pd.DataFrame(rows)
print(f6d.round(3).to_string(index=False))
print("\nper session, Methods variance, first puff, code epochs (reading without sub-5):")
print(per_session["without sub-5"].round(3).to_string())

**Result: reproduced, resting on the assumption that the excluded animal is sub-5.** On the criterion's main reading (per-unit variance, first puff, code epochs) without sub-5, the projection is 0.867 (Pre), 0.007 (Ketamine) and 0.511 (Post), all within tolerance, with corrected p = 0.001 (Pre against Ketamine) and 0.032 (Post against Ketamine). Without sub-3 it also holds (q = 0.001 and 0.012); without sub-1 the Post comparison misses (q = 0.051), which would make it partially reproduced. With all eight puffs the Pre mean falls to 0.45, outside tolerance, and with the Methods-literal epochs the Post comparison fails in two of three readings.

## fig6f-mouse-fast-unchanged

> In mice, activity along the fast (0 to 70 ms) coding dimension was not changed by ketamine.

(Figure 6F; table S3, Fig. 6F)

Criterion: weights from each Pre puff, 0 to 70 ms against the 1 s before (Methods) or the 70 ms before (code); projection over all series and puffs, z-scored over -1 to 2 s, minus one mean over sessions of the -1 to 0 s bins (as the code), quantified over 0 to 70 ms.


In [ ]:
rows = []
for rd, ks in READINGS.items():
    for var in ("unit", "pooled"):
        for ref in ("1s", "70ms"):
            tr = {c: np.array([scipy.stats.zscore(res[k]["cd_proj"][("fast", var, ref, cc)]["all"]) for k in ks])
                  for c, cc in (("Pre", "pre"), ("Ketamine", "ket"), ("Post", "post"))}
            vals = {}
            for c, a in tr.items():
                a = a - a[:, :100].mean()           # analysis.py subtracts one mean over sessions and baseline bins
                vals[c] = a[:, 100:107].mean(1)
            t1 = scipy.stats.ttest_rel(vals["Pre"], vals["Ketamine"])
            t2 = scipy.stats.ttest_rel(vals["Post"], vals["Ketamine"])
            q = bh([t1.pvalue, t2.pvalue])
            rows.append({"reading": rd, "variance": var, "reference": ref, "Pre": vals["Pre"].mean(), "Ket": vals["Ketamine"].mean(),
                         "Post": vals["Post"].mean(), "t Pre-Ket": t1.statistic, "q Pre-Ket": q[0], "t Post-Ket": t2.statistic,
                         "q Post-Ket": q[1], "Pre-Ket diff": (vals["Pre"] - vals["Ketamine"]).mean()})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** The paper's test finds no Pre against Ketamine difference in any reading: corrected p from 0.27 to 0.98 (sub-5 excluded, per-unit variance, 1 s reference: 4.23 against 4.18, q = 0.69). The Pre means are 4.0 to 4.47, below the expected 4.48 +- 0.25 with the per-unit variance and within it with the pooled variance (4.43 to 4.47 without sub-5).

## fig6b-persistent-dimension-distributed

> The neurons carrying the persistent peri-puff dimension were not focally localized but distributed over much of the brain.

(Figure 6B; fig. S36A)

The top 10% of units by absolute persistent weight per session (fig6d-mouse-persistent weights), placed in the brainwide-npix top-level divisions through the Allen tree.


In [ ]:
for var in ("unit", "pooled"):
    for rd, ks in READINGS.items():
        tops = np.hstack([res[k]["top_units_regions"][var] for k in ks])
        incl = np.hstack([res[k]["regions"] for k in ks])
        top_tl = pd.Series([K.top_level(r, paths) for r in tops]).value_counts()
        inc_tl = pd.Series([K.top_level(r, paths) for r in incl]).value_counts()
        tab = pd.DataFrame({"included units": inc_tl, "top-10% units": top_tl}).fillna(0).astype(int)
        tab["share of top units"] = (tab["top-10% units"] / tab["top-10% units"].sum()).round(3)
        div = [d for d in tab.index if d != "NA" and tab.loc[d, "included units"] >= 30]
        missing = [d for d in div if tab.loc[d, "top-10% units"] == 0]
        print(f"{var} variance, {rd}: divisions with >= 30 included units: {div}; without top units: {missing}; "
              f"largest share {tab.drop('NA', errors='ignore')['share of top units'].max():.3f}")
        if var == "unit" and rd == "without sub-5":
            print(tab.to_string())

**Result: reproduced.** In every reading each top-level division with at least 30 included units holds top-10% units, and no division holds more than 0.32 of them (0.21 for Isocortex with per-unit variance, sub-5 excluded).

## fig6h-mouse-timescale

> In mice, the intrinsic timescale of the persistent dimension in the 30 s before each series was shorter during ketamine (0.14 s) than before (0.45 s).

(Figure 6H; fig. S34B; table S3, Fig. 6H)

Criterion: the 30 s before the first puff of Pre series 11 to 20 and Ketamine series 23 to 32 projected on the persistent weights; analysis.py extract_timescale (autocorrelation from lag 10 ms, normalized to its maximum, fit over 2.5 s with initial a=1, b=0.2, c=0); (a) per series fits averaged within session, (b) one fit to the concatenated series. A fit that does not converge is left out of the session mean and counted.


In [ ]:
rows = []
for rd, ks in READINGS.items():
    for var in ("unit", "pooled"):
        for pre_sl, pre_lab in ((slice(10, 20), "Pre 11-20"),):
            a_pre, a_ket, b_pre, b_ket, nfail = [], [], [], [], 0
            for k in ks:
                ps = res[k]["base_sigs"][var]["pre_sigs"][pre_sl]
                kt = res[k]["base_sigs"][var]["ket_sigs"]
                tp = np.array([K.extract_timescale(s) for s in ps]); tk = np.array([K.extract_timescale(s) for s in kt])
                nfail += np.isnan(tp).sum() + np.isnan(tk).sum()
                a_pre.append(np.nanmean(tp)); a_ket.append(np.nanmean(tk))
                b_pre.append(K.extract_timescale(ps.reshape(-1))); b_ket.append(K.extract_timescale(kt.reshape(-1)))
            for lab, p_, k_ in (("(a) per series", a_pre, a_ket), ("(b) concatenated", b_pre, b_ket)):
                p_, k_ = np.array(p_), np.array(k_)
                t = scipy.stats.ttest_rel(p_, k_)
                rows.append({"reading": rd, "variance": var, "combine": lab, "Pre": p_.mean(), "Ket": k_.mean(), "t": t.statistic, "p": t.pvalue,
                             "within": abs(p_.mean() - 0.45) <= 0.14 and abs(k_.mean() - 0.14) <= 0.12, "failed fits": nfail if lab.startswith("(a)") else 0,
                             "Pre range": f"{p_.min():.3f}-{p_.max():.3f}", "Ket range": f"{k_.min():.3f}-{k_.max():.3f}"})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced, resting on the assumption that the excluded animal is sub-5.** Without sub-5 and with per-unit variance, per-series fits give 0.484 s (Pre) and 0.200 s (Ketamine), p = 0.0097, and one concatenated fit gives 0.388 and 0.119 s, p = 0.0041, all within tolerance. Without sub-1 or sub-3 the per-series reading fails (p = 0.56 and 0.48): a sub-5 session then enters with a per-series Ketamine mean of 1.71 s (1075 s with pooled weights, a fit that runs away); the concatenated reading holds (p = 0.012 and 0.010).

## fig6j-mouse-coupling

> In mice, correlations among the most persistent neurons in the 30 s before each series fell during ketamine to 0.49 of their preinfusion value.

(Figure 6J; table S3, Fig. 6J)

Criterion: (a) Methods, units in the top 10% of absolute persistent weight, Pearson r of z-scored rates over each 30 s window before the first puff, mean over edges, averaged over series (Pre 1 to 10, Ketamine 23 to 32); (b) code, units whose signed weight exceeds the 90th percentile of absolute weights, median absolute r over the concatenated windows. A session with no value (a silent unit leaves no defined correlation) is left out of the test and counted. Ketamine divided by Pre within session; paired t-test of Pre against Ketamine.


In [ ]:
rows = []
for rd, ks in READINGS.items():
    for var in ("unit", "pooled"):
        for lab, sname, meas in (("(a) Methods", "methods", "per_series_mean_r"), ("(b) code", "code", "concat_median_abs_r")):
            pre_v, ket_v, nu = [], [], []
            for k in ks:
                c = res[k]["coupling"][(var, sname)]
                f = (lambda d: np.nanmean(d[meas])) if meas == "per_series_mean_r" else (lambda d: d[meas])
                pre_v.append(f(c["pre"])); ket_v.append(f(c["ket"])); nu.append(c["n_units"])
            pre_v, ket_v = np.array(pre_v), np.array(ket_v)
            ok = ~(np.isnan(pre_v) | np.isnan(ket_v))
            t = scipy.stats.ttest_rel(pre_v[ok], ket_v[ok])
            ratio = ket_v[ok] / pre_v[ok]
            rows.append({"reading": rd, "variance": var, "measure": lab, "Pre": pre_v.mean(), "Ket": ket_v.mean(),
                         "Ket/Pre mean": ratio.mean(), "ratio range": f"{ratio.min():.2f}-{ratio.max():.2f}", "t": t.statistic, "p": t.pvalue,
                         "neurons per session": f"{min(nu)}-{max(nu)}", "sessions without a value": int((~ok).sum())})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: partially reproduced.** Without sub-5, the code's reading with the code's weights (pooled variance) gives Ketamine at 0.56 of Pre, within tolerance, p = 0.0012, and with per-unit weights 0.74, p = 0.021. The Methods reading with per-unit weights gives 0.82, p = 0.058, which misses; with pooled weights it gives 0.59, p = 0.0039. Coupling falls under every reading, but not significantly under the Methods reading with the Methods weights, so the criterion's "under both" is not met.

## fig5d-emotion-dimension-accumulates

> Population activity along an emotion-like dimension (after the series against before it) rose and saturated across the puff series and decayed over tens of seconds afterwards; under ketamine it did not accumulate.

(Figure 5C and 5D; table S3, Fig. 5D)

Criterion: per session from the puff-modulated units; target the 1 s from 5 s after the last puff onset (Methods; the code's 3 to 4 s as a variant), reference the 750 ms ending 250 ms before the first puff (-1 to -0.25 s), weights normalized by per-unit variance (Methods; pooled, code variant), orthogonalized by analysis.py my_qr to the eye-reflex dimension of the last four puffs (200 ms after against 200 ms before), signed so that puff 8 exceeds puff 1. Projections are averaged over series, baseline-subtracted with the 250 ms before the first puff. The series windows are the eight -1 to 2 s puff windows (-1 to 23 s after the first puff); the post-series window starts 1 s before the last puff onset and lasts 50 s. Projections under the unit vector (Methods) and the sum of absolute values (code) are both shown. Intervals are mean +- 1.96 s.e.m. across sessions.


In [ ]:
def emo_signal(o, cond, scale):
    """Projection over the series (8 puff windows, -1 to 23 s after the first puff) and the post-series window
    (from 1 s before the last puff, 50 s), baseline-subtracted with the 250 ms before the first puff."""
    f = o["l1_factor"] if scale == "l1" else 1.0
    s8 = o[cond]["s8"] * f
    post = o[cond]["post"] * f
    b = s8[75:100].mean()
    return s8 - b, post - b

def emo_checks(ks, group="all", tw="methods", var="unit", scale="unit"):
    d = {}
    for cond in ("pre", "ket"):
        s8 = []; post = []
        for k in ks:
            o = res[k]["emotion"].get((group, tw, var))
            if o is None:
                continue
            a, b = emo_signal(o, cond, scale)
            s8.append(a); post.append(b)
        d[cond] = (np.array(s8), np.array(post))
    s8p, postp = d["pre"]; s8k, postk = d["ket"]
    n = len(s8p)
    v8 = s8p[:, 7 * 300 + 75:7 * 300 + 100].mean(1)
    v2 = s8p[:, 300 + 75:300 + 100].mean(1)
    v10 = postp[:, 1100:1200].mean(1)          # 10 to 11 s after the last puff onset
    vend = postp[:, 4500:5000].mean(1)         # last 5 s of the 45 s after the series
    k8 = s8k[:, 7 * 300 + 75:7 * 300 + 100].mean(1)
    c8, c2, c10, cend, ck8 = ci95(v8), ci95(v2), ci95(v10), ci95(vend), ci95(k8)
    ok = {"pre8>0": c8[1] > 0, "pre8>pre2": c8[0] > c2[0], "10-11s>0": c10[1] > 0,
          "end includes 0": cend[1] <= 0 <= cend[2], "ket8 includes 0 or below pre": (ck8[1] <= 0 <= ck8[2]) or ck8[2] < c8[0]}
    return n, {"before puff 2": c2, "before puff 8": c8, "10-11 s after": c10, "last 5 s": cend, "ket before puff 8": ck8}, ok

rows = []
for rd, ks in READINGS.items():
    for tw in ("methods", "code"):
        for var in ("unit", "pooled"):
            for scale in ("unit", "l1"):
                n, c, ok = emo_checks(ks, "all", tw, var, scale)
                row = {"reading": rd, "target": tw, "variance": var, "scale": scale, "n": n}
                row.update({k: f"{v[0]:.3f} [{v[1]:.3f}, {v[2]:.3f}]" for k, v in c.items()})
                row.update(ok); row["all hold"] = all(ok.values())
                rows.append(row)
f5d = pd.DataFrame(rows)
print(f5d.to_string(index=False))

**Result: partially reproduced.** In every reading the Pre projection before puff 8 is above zero and above that before puff 2 (Methods reading, sub-5 excluded: 1.033 [0.767, 1.300] against 0.255), it is still above zero 10 to 11 s after the last puff (0.790 [0.609, 0.971]), and under ketamine the value before puff 8 includes zero (-0.018 [-0.203, 0.167]). It does not return to baseline: the last 5 s of the 45 s after the series stay above zero in every reading (0.303 [0.203, 0.403]).

## fig5e-late-reduced

> On the first puff of a series, late emotion-like activity (2 to 3 s after onset) was reduced by ketamine (0.546 to 0.005).

(Figure 5E; table S3, Fig. 5E)

Criterion: per session the first-puff signal of analysis.py plot_affective_firstpuff_persist (series window from 0.25 s before the first puff, then the post-series window from 3 s after the last puff), minus its first 250 ms; early 0 to 0.25 s, late 2 to 3 s after the first puff; two-sided paired t-tests, Benjamini-Hochberg over the two windows.


In [ ]:
def first_puff(ks, tw, var, scale):
    """analysis.py plot_affective_firstpuff_persist: signal from 0.25 s before the first puff, minus its first 250 ms."""
    out = {}
    for cond in ("pre", "ket"):
        e, l = [], []
        for k in ks:
            o = res[k]["emotion"][("all", tw, var)]
            f = o["l1_factor"] if scale == "l1" else 1.0
            sig = np.concatenate([o[cond]["s8"][75:], o[cond]["post"][400:5000]]) * f
            sig = sig - sig[:25].mean()
            e.append(sig[25:50].mean()); l.append(sig[225:325].mean())
        out[cond] = (np.array(e), np.array(l))
    return out

rows5e = []
for rd, ks in READINGS.items():
    for tw in ("methods", "code"):
        for var in ("unit", "pooled"):
            for scale in ("unit", "l1"):
                fp = first_puff(ks, tw, var, scale)
                te = scipy.stats.ttest_rel(fp["pre"][0], fp["ket"][0])
                tl = scipy.stats.ttest_rel(fp["pre"][1], fp["ket"][1])
                q = bh([te.pvalue, tl.pvalue])
                rows5e.append({"reading": rd, "target": tw, "variance": var, "scale": scale,
                               "early Pre": fp["pre"][0].mean(), "early Ket": fp["ket"][0].mean(), "t early": te.statistic, "q early": q[0],
                               "late Pre": fp["pre"][1].mean(), "late Ket": fp["ket"][1].mean(), "t late": tl.statistic, "q late": q[1],
                               "late within": abs(fp["pre"][1].mean() - 0.546) <= 0.34 and abs(fp["ket"][1].mean() - 0.005) <= 0.084,
                               "early within": abs(fp["pre"][0].mean() - 0.601) <= 0.55 and abs(fp["ket"][0].mean() - 0.332) <= 0.27})
f5e = pd.DataFrame(rows5e)
print(f5e.round(4).to_string(index=False))

**Result: not reproduced.** On the criterion's main reading (Methods target window, per-unit variance) the late projection falls from 0.30 to 0.03 (sub-5 excluded; 0.31 to 0.04 and 0.36 to 0.04 under the other exclusions), both within tolerance under the unit-vector scaling, but the corrected p is 0.105, 0.109 and 0.113 under the three exclusions, above 0.05. The code's variants pass: its 3 to 4 s target window gives q = 0.015 to 0.023 with per-unit variance, and the pooled variance with the Methods window gives q = 0.034 to 0.073.

## fig5e-rapid-unchanged

> The rapid emotion-like response to the first puff (0 to 0.25 s) did not differ between preinfusion and ketamine.

(Figure 5E; table S3, Fig. 5E)

The early window of the computation above.


In [ ]:
print(f5e[["reading", "target", "variance", "scale", "early Pre", "early Ket", "t early", "q early", "early within"]].round(4).to_string(index=False))

**Result: reproduced.** On the main reading the early projection is 0.80 against 0.33 (sub-5 excluded), corrected p = 0.105 (0.109 and 0.113 under the other exclusions), so the paper's test finds no difference. Both means are within tolerance under the unit-vector scaling. Under the code's 3 to 4 s target window the early difference is significant (q = 0.023 to 0.048).

## figs32c-regional-emotion-dimension

> Emotion-like dimensions computed within each high-level brain area all showed accumulating persistent dynamics before ketamine that ketamine abolished.

(Figure S32C)

The emotion-like dimension recomputed within each brainwide-npix top-level division for every session with at least 20 puff-modulated units in it (Methods target window and per-unit variance; the code's window as a variant). Areas with at least two such sessions are judged on the puff-8 conditions of fig5d-emotion-dimension-accumulates.


In [ ]:
rows_r = []
for rd, ks in READINGS.items():
    for tw in ("methods", "code"):
        for area in K.TOP_LEVEL:
            qual = [k for k in ks if (area, tw, "unit") in res[k]["emotion"]]
            if len(qual) < 2:
                continue
            n, c, ok = emo_checks(qual, area, tw, "unit", "unit")
            rows_r.append({"reading": rd, "target": tw, "area": area, "sessions": n,
                           "pre before puff 8": f"{c['before puff 8'][0]:.3f} [{c['before puff 8'][1]:.3f}, {c['before puff 8'][2]:.3f}]",
                           "pre before puff 2": f"{c['before puff 2'][0]:.3f}",
                           "ket before puff 8": f"{c['ket before puff 8'][0]:.3f} [{c['ket before puff 8'][1]:.3f}, {c['ket before puff 8'][2]:.3f}]",
                           "meets": ok["pre8>0"] and ok["pre8>pre2"] and ok["ket8 includes 0 or below pre"]})
f32 = pd.DataFrame(rows_r)
print(f32.to_string(index=False))
print(f32.groupby(["reading", "target"])["meets"].agg(["sum", "count"]))

**Result: reproduced.** All seven areas with at least two qualifying sessions (Isocortex, OLF, HPF, STR, TH, HY, MB) meet the puff-8 conditions under every exclusion and both target windows; for example without sub-5 (Methods window) Isocortex is 0.533 [0.333, 0.733] before puff 8 against 0.180 before puff 2, and 0.008 [-0.096, 0.113] under ketamine.

## fig5g-tau-persist-model

> A two-phase first-order model in which only the persistence timescale differs between preinfusion and ketamine closely matched the emotion-like population activity.

(Figure 5F and 5G; table S3, Fig. 5G)

Criterion: analysis.py model_system and theoretical_persist (inputs of 250 ms every 3 s, the model output from -0.25 to 23 s and 25 to 70 s after the first puff), fitted with curve_fit (bounds [0, 100], factor per condition in [0, 1], null factor in [0.99999, 1]) to the across-session mean of the Pre and Ketamine projections over the same windows, concatenated. tauP Pre and tauP Ketamine are tauP times each condition's factor.


In [ ]:
def model_data(ks, tw, var, scale):
    segs = []
    for cond in ("pre", "ket"):
        a = []
        for k in ks:
            s8, post = emo_signal(res[k]["emotion"][("all", tw, var)], cond, scale)
            a.append(np.concatenate([s8[75:2400], post[500:5000]]))     # -0.25 to 23 s and 25 to 70 s after the first puff
        segs.append(np.mean(a, 0))
    return np.concatenate(segs)

import time as _time
rows_m = []
fits = {}
t0 = _time.time()
for rd, ks in READINGS.items():
    for tw in ("methods", "code"):
        for var in ("unit", "pooled"):
            for scale in ("unit", "l1"):
                data = model_data(ks, tw, var, scale)
                r = {}
                for kind in ("persist", "saturation", "broadcast", "symmetric"):
                    try:
                        r[kind] = K.compare_model(kind, data)
                    except Exception as e:
                        r[kind] = {"varexp": np.nan, "fit": [np.nan] * 5, "pred": np.full_like(data, np.nan), "null": np.full_like(data, np.nan), "error": str(e)}
                fits[(rd, tw, var, scale)] = (data, r)
                pm = r["persist"]
                rows_m.append({"reading": rd, "target": tw, "variance": var, "scale": scale, "VE persist": pm["varexp"],
                               "S": pm["fit"][0], "tauB": pm["fit"][1], "tauP": pm["fit"][2], "factor Pre": pm["fit"][3], "factor Ket": pm["fit"][4],
                               "tauP Pre": pm["fit"][2] * pm["fit"][3], "tauP Ket": pm["fit"][2] * pm["fit"][4]})
print(f"{len(fits)} fits in {_time.time() - t0:.0f} s")
print(pd.DataFrame(rows_m).round(4).to_string(index=False))

**Result: reproduced.** In all 24 readings the variance explained is positive (0.81 to 0.95) and the fitted Ketamine persistence timescale is far shorter than the Pre one (Methods reading, sub-5 excluded: 32.2 s against 0.25 s, variance explained 0.866).

## figs33-alternative-models

> Alternative models in which ketamine changes the input scale S or the broadcast rate, or a symmetric single-timescale model, fit the ketamine puff-series activity worse than the model in which ketamine changes the persistence timescale.

(Figure S33A to S33D; Results, Brain-wide activity screen during emergence of affect in mice)

The same fits with the factor on S (theoretical_saturation), on tauB (theoretical_broadcast) or a single timescale (theoretical_symmetric), and the variance explained on the Ketamine half of the concatenated data (each model against its own null fit).


In [ ]:
rows_a = []
for key, (data, r) in fits.items():
    half = len(data) // 2
    ve = {}
    for kind, m in r.items():
        ve[kind] = 1 - np.mean((data[half:] - m["pred"][half:]) ** 2) / np.mean((data[half:] - m["null"][half:]) ** 2)
    best = max(ve, key=lambda k: -np.inf if np.isnan(ve[k]) else ve[k])
    rows_a.append(dict(zip(("reading", "target", "variance", "scale"), key), **{f"VE ket {k}": v for k, v in ve.items()}, best=best))
print(pd.DataFrame(rows_a).round(4).to_string(index=False))

**Result: reproduced.** On the Ketamine segment the tauP model explains the most variance in 22 of 24 readings, including every reading with the Methods target window (sub-5 excluded, Methods: 0.884 against 0.869 for S, 0.868 for tauB and 0.456 for the symmetric model). Its margin over the S and tauB models is small (0.0002 to 0.04), and with the code's window and pooled variance without sub-3 the S model is best (0.9573 against 0.9565, and 0.9619 against 0.9556 with the L1 scaling).

## figs34d-tone-fast-unchanged

> Fast tone-related population activity (0 to 70 ms) was not changed by ketamine.

(Figure S34D; table S3, fig. S34 D)

Tone windows of the puff-modulated units. Splits: 'infusion' takes the 10 tone series before the infusion time in the file as Pre and counts Ketamine (series 2 to 6) and Post (8 to 10) from the infusion, as the criterion reads the code; 'code' splits at the code's ketamine onset (30 s after the last puff of series 20), which falls before the infusion, so that the code's Pre has 9 tone series and its first "post" series precedes the infusion. Otherwise as fig6f-mouse-fast-unchanged.


In [ ]:
def tone_vals(ks, split, key, puffs, win, base="global"):
    out = {}
    for c in ("pre", "ket", "post"):
        a = np.array([scipy.stats.zscore(res[k]["tone"][split]["proj"][key + (c,)][puffs]) for k in ks])
        a = a - (a[:, :100].mean() if base == "global" else a[:, :100].mean(1, keepdims=True))
        out[c] = a[:, win].mean(1)
    return out

def tone_tests(v):
    t1 = scipy.stats.ttest_rel(v["pre"], v["ket"]); t2 = scipy.stats.ttest_rel(v["post"], v["ket"])
    q = bh([t1.pvalue, t2.pvalue])
    return t1.statistic, q[0], t2.statistic, q[1]

rows = []
for rd, ks0 in READINGS.items():
    ks = [k for k in ks0 if k in TONE_SESSIONS]
    for split in ("infusion", "code"):
        for var in ("unit", "pooled"):
            for ref in ("1s", "70ms"):
                v = tone_vals(ks, split, ("fast", var, ref), "all", slice(100, 107))
                t1, q1, t2, q2 = tone_tests(v)
                rows.append({"reading": rd, "sessions": len(ks), "split": split, "variance": var, "reference": ref,
                             "Pre": v["pre"].mean(), "Ket": v["ket"].mean(), "Post": v["post"].mean(),
                             "t Pre-Ket": t1, "q Pre-Ket": q1, "t Post-Ket": t2, "q Post-Ket": q2})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced, resting on the assumption that the excluded animal is sub-5.** Without sub-5 (7 tone sessions from 7 mice, as table S3) the paper's test finds no Pre against Ketamine difference in any reading (corrected p 0.14 to 0.39; main reading 2.71 against 1.37, q = 0.14). With sub-1 or sub-3 excluded instead (8 sessions), the difference is significant in some readings (q = 0.028 and 0.033 on the main reading), which would not reproduce it.

## figs34e-tone-persistent

> Persistent tone-related activity (350 to 500 ms) fell during ketamine (1.531 to 0.392) and recovered (1.385).

(Figure S34E; table S3, fig. S34 E)

Weights from Pre tone series means over 150 to 350 ms against the 1 s before the tone; projection over all eight tones, z-scored over -1 to 2 s, minus its mean over -1 to 0 s, quantified over 350 to 500 ms. Paired t-tests, Benjamini-Hochberg over the two.


In [ ]:
rows = []
for rd, ks0 in READINGS.items():
    ks = [k for k in ks0 if k in TONE_SESSIONS]
    for split in ("infusion", "code"):
        for var in ("unit", "pooled"):
            v = tone_vals(ks, split, ("persist", var), "all", slice(135, 150), base="own")
            t1, q1, t2, q2 = tone_tests(v)
            m = [v[c].mean() for c in ("pre", "ket", "post")]
            rows.append({"reading": rd, "sessions": len(ks), "split": split, "variance": var, "Pre": m[0], "Ket": m[1], "Post": m[2],
                         "t Pre-Ket": t1, "q Pre-Ket": q1, "t Post-Ket": t2, "q Post-Ket": q2,
                         "holds": q1 < 0.05 and q2 < 0.05 and m[1] < min(m[0], m[2]),
                         "within": abs(m[0] - 1.531) <= 0.16 and abs(m[1] - 0.392) <= 0.35 and abs(m[2] - 1.385) <= 0.66})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** On the main reading (infusion split, per-unit variance) without sub-5, the projection is 1.639 (Pre), 0.467 (Ketamine) and 1.500 (Post), all within tolerance, with corrected p = 0.0004 and 0.028. Every exclusion and split with per-unit variance gives the same; two pooled-variance readings miss on the Post comparison (q = 0.054 and 0.055).

## figs34f-tone-shorter-than-puff

> Persistent responses to the neutral tone were shorter than those to the eyepuff.

(Figure S34F; table S3, fig. S34 F)

analysis.py compare_tone_puff_late_proj: Pre projections on each stimulus's persistent dimension, averaged over the eight stimuli of a series, turned into d' (calculate_dprime, the series as trials) against the 1 s before onset. Half-peak fall is the time from onset until the across-session mean d' falls below half its peak after the peak.


In [ ]:
def half_fall(tr):
    post = tr[100:]
    pk = int(np.argmax(post))
    below = np.where(post[pk:] < 0.5 * post[pk])[0]
    return (pk + below[0]) * K.BIN if len(below) else np.nan

rows = []
for rd, ks0 in READINGS.items():
    ks = [k for k in ks0 if k in TONE_SESSIONS]
    for split in ("infusion", "code"):
        for var in ("unit", "pooled"):
            dp = np.array([res[k]["tone"][split]["dprime_puff"][var] for k in ks])
            dt = np.array([res[k]["tone"][split]["dprime_tone"][var] for k in ks])
            cp, ct = ci95(dp[:, 200:300].mean(1)), ci95(dt[:, 200:300].mean(1))
            rows.append({"reading": rd, "sessions": len(ks), "split": split, "variance": var,
                         "puff half-peak fall (s)": half_fall(dp.mean(0)), "tone half-peak fall (s)": half_fall(dt.mean(0)),
                         "puff d' 1-2 s": f"{cp[0]:.2f} [{cp[1]:.2f}, {cp[2]:.2f}]", "tone d' 1-2 s": f"{ct[0]:.2f} [{ct[1]:.2f}, {ct[2]:.2f}]",
                         "holds": half_fall(dp.mean(0)) > half_fall(dt.mean(0)) and cp[1] > ct[2]})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** In every reading the mean puff d' falls to half its peak 0.90 to 0.99 s after onset against 0.16 to 0.17 s for the tone, and the puff d' over 1 to 2 s (1.04 to 1.10) lies above the tone's (0.09 to 0.21) with non-overlapping intervals.

## figs34g-tone-timescale

> The baseline intrinsic timescale of the persistent tone population was shorter during ketamine (0.089 s) than before (0.268 s).

(Figure S34G; table S3, fig. S34 G)

As fig6h-mouse-timescale with the tone dimension and the 30 s before the first tone of each series; Pre tone series 5 to 10 (code) or all 10 (table S3), Ketamine tone series 2 to 6.


In [ ]:
rows = []
for rd, ks0 in READINGS.items():
    ks = [k for k in ks0 if k in TONE_SESSIONS]
    for split in ("infusion", "code"):
        for var in ("unit", "pooled"):
            for pre_lab, pre_sl in (("Pre 5-10 (code)", slice(4, 10)), ("Pre all 10 (table S3)", slice(0, 10))):
                a_p, a_k, b_p, b_k = [], [], [], []
                for k in ks:
                    bs = res[k]["tone"][split]["base_sigs"][var]
                    ps, kt = bs["pre_sigs"][pre_sl], bs["post_sigs"][1:6]
                    a_p.append(np.nanmean([K.extract_timescale(s) for s in ps])); a_k.append(np.nanmean([K.extract_timescale(s) for s in kt]))
                    b_p.append(K.extract_timescale(ps.reshape(-1))); b_k.append(K.extract_timescale(kt.reshape(-1)))
                for lab, p_, k_ in (("(a) per series", a_p, a_k), ("(b) concatenated", b_p, b_k)):
                    p_, k_ = np.array(p_), np.array(k_)
                    t = scipy.stats.ttest_rel(p_, k_)
                    rows.append({"reading": rd, "sessions": len(ks), "split": split, "variance": var, "pre series": pre_lab, "combine": lab,
                                 "Pre": p_.mean(), "Ket": k_.mean(), "t": t.statistic, "p": t.pvalue,
                                 "holds": t.pvalue < 0.05 and k_.mean() < p_.mean(),
                                 "within": abs(p_.mean() - 0.268) <= 0.073 and abs(k_.mean() - 0.089) <= 0.024})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: partially reproduced, resting on the assumption that the excluded animal is sub-5.** Without sub-5 the Ketamine timescale is shorter in every reading (p = 0.0008 to 0.037). The concatenated fits are within tolerance (main reading 0.283 against 0.082 s, p = 0.0077), but the per-series fits put Ketamine at 0.140 to 0.161 s, outside 0.089 +- 0.024. With sub-1 or sub-3 excluded the per-series reading fails the test (p = 0.24 to 0.91).

## figs34h-tone-coupling

> Correlations among the persistent tone neurons before the tone series fell during ketamine to 0.460 of their preinfusion value.

(Figure S34H; table S3, fig. S34 H)

As fig6j-mouse-coupling with the tone dimension and the 30 s before each tone series (Pre tone series, Ketamine 2 to 6). The code's own loop takes the windows before puff series 1 to 10 and 23 to 32 for any stimulus; that is shown as (b').


In [ ]:
rows = []
for rd, ks0 in READINGS.items():
    ks = [k for k in ks0 if k in TONE_SESSIONS]
    for split in ("infusion", "code"):
        for var in ("unit", "pooled"):
            for lab, sname, meas, cp, ck in (("(a) Methods", "methods", "per_series_mean_r", "pre", "ket"),
                                             ("(b) code", "code", "concat_median_abs_r", "pre", "ket"),
                                             ("(b') code, puff windows", "code", "concat_median_abs_r", "pre_puffwin", "ket_puffwin")):
                f = (lambda d: np.nanmean(d[meas])) if meas == "per_series_mean_r" else (lambda d: d[meas])
                pv = np.array([f(res[k]["tone"][split]["coupling"][(var, sname)][cp]) for k in ks])
                kv = np.array([f(res[k]["tone"][split]["coupling"][(var, sname)][ck]) for k in ks])
                ok = ~(np.isnan(pv) | np.isnan(kv))
                t = scipy.stats.ttest_rel(pv[ok], kv[ok])
                ratio = kv[ok] / pv[ok]
                rows.append({"reading": rd, "split": split, "variance": var, "measure": lab, "sessions": int(ok.sum()),
                             "Ket/Pre mean": ratio.mean(), "t": t.statistic, "p": t.pvalue,
                             "holds": t.pvalue < 0.05 and kv[ok].mean() < pv[ok].mean(), "within": abs(ratio.mean() - 0.46) <= 0.16})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: not reproduced.** With the 30 s before each tone series, neither reading finds a significant fall without sub-5: the Methods reading gives Ketamine at 1.23 of Pre (p = 0.90) and the code's measure 0.82 (p = 0.18) with per-unit weights, and 0.87 (p = 0.40) and 0.66 (p = 0.061) with pooled weights. The code's own loop, which takes the windows before puff series 1 to 10 and 23 to 32, gives 0.53 (p = 0.019, pooled weights) and 0.67 (p = 0.051, per-unit weights), near the paper's 0.46.

## figs30-correlation-structure

> Ketamine changed the correlation structure of single units at rest, increasing correlations within regions such as retrosplenial/posteromedial and orbitofrontal cortex while reducing correlations between regions.

(Figure S30A; Results, Brain-wide activity screen during emergence of affect in mice)

Criterion: Pearson correlations of the smoothed rates of all units over the whole Pre period (from the start of the recording to the code's ketamine onset, 30 s after the last puff of series 20) and over the 10 minutes from that onset; per session the mean change (Ketamine minus Pre) over pairs in the same finest region and over pairs in different regions (`NA` left out). Pairs with a unit silent in a period have no correlation and are counted. RSP and ORB within-region changes: pairs in the same finest region under RSP or ORB; the change over all pairs within the RSP or ORB family is also shown. t-tests across the 13 sessions.


In [ ]:
rows = []
for k in names:
    reg = res[k]["regions_all"]
    ok = reg != "NA"
    d = (res[k]["corr_ket"] - res[k]["corr_pre"])[np.ix_(ok, ok)].astype(float)
    r = reg[ok]
    same = r[:, None] == r[None, :]
    iu = np.triu(np.ones_like(same), 1).astype(bool)
    fin = np.isfinite(d)
    row = {"session": k, "units": int(ok.sum()), "within change": np.nanmean(d[same & iu & fin]), "between change": np.nanmean(d[~same & iu & fin]),
           "pairs dropped (silent unit)": int((iu & ~fin).sum())}
    for fam in ("RSP", "ORB"):
        m = np.array([K.under(x, fam, paths) for x in r])
        row[f"{fam} units"] = int(m.sum())
        row[f"{fam} within change"] = np.nanmean(d[np.outer(m, m) & same & iu & fin]) if m.sum() > 1 else np.nan
        row[f"{fam} family change"] = np.nanmean(d[np.outer(m, m) & iu & fin]) if m.sum() > 1 else np.nan
    rows.append(row)
c30 = pd.DataFrame(rows).set_index("session")
print(c30.round(4).to_string())
tw = scipy.stats.ttest_1samp(c30["within change"], 0); tb = scipy.stats.ttest_1samp(c30["between change"], 0)
tp = scipy.stats.ttest_rel(c30["within change"], c30["between change"])
print(f"within-region change: mean {c30['within change'].mean():.4f}, t = {tw.statistic:.2f}, p = {tw.pvalue:.4f}")
print(f"between-region change: mean {c30['between change'].mean():.4f}, t = {tb.statistic:.2f}, p = {tb.pvalue:.4f}")
print(f"within against between (paired): t = {tp.statistic:.2f}, p = {tp.pvalue:.4f}")
for fam in ("RSP", "ORB"):
    for col in ("within change", "family change"):
        v = c30[f"{fam} {col}"].dropna()
        print(f"{fam} {col}: positive in {(v > 0).sum()} of {len(v)} sessions recording it")

**Result: partially reproduced.** Within-region correlations rise in RSP in 9 of 10 sessions that record it and in ORB in 5 of 7. Across the 13 sessions the mean within-region change is +0.0063 (p = 0.052), short of significance, and the between-region change is -0.0001 (p = 0.89), not below zero. Within-region changes exceed between-region changes (paired p = 0.016), a comparison the criterion does not use.

## figs29-rsp-oscillations

> Ketamine brought low-frequency oscillations into the population activity, especially strong in retrosplenial cortex.

(Figure S29C and S29D; Results, Brain-wide activity screen during emergence of affect in mice)

Criterion: scipy.signal.welch (100 Hz, nperseg 2048) of each unit's z-scored rate over one minute, z-scored across frequency, peak below 7.5 Hz; the minute before the infusion time in the file against the minute from 5 min and from 10 min after it. Regions merged across layers (reduce_region); RSP is any region under RSP. Units silent in a window have no spectrum and are left out.


In [ ]:
rows = []
for k in names:
    reg = np.array([K.reduce_region(x) for x in res[k]["regions_all"]])
    for after in ("after5", "after10"):
        ch = res[k]["spec_peak"][after] - res[k]["spec_peak"]["before"]
        for r in np.unique(reg):
            m = reg == r
            rows.append({"session": k, "after": after, "region": r, "rsp": K.under(r, "RSP", paths), "units": int(m.sum()), "change": np.nanmean(ch[m]),
                         "values": ch[m]})
sp = pd.DataFrame(rows)
sp = sp[sp.region != "NA"]
for after in ("after5", "after10"):
    s = sp[sp.after == after]
    rsp_sess = s[s.rsp].groupby("session").apply(lambda g: np.nanmean(np.concatenate(g["values"].to_list())))
    pooled = s.groupby("region").apply(lambda g: pd.Series({"units": g["units"].sum(), "change": np.nanmean(np.concatenate(g["values"].to_list()))}))
    pooled = pooled[pooled.units >= 30].sort_values("change", ascending=False)
    print(f"\n{after}: RSP units' mean change in z-scored low-frequency peak per session recording RSP:")
    print(rsp_sess.round(3).to_string())
    print(f"rises in {(rsp_sess > 0).sum()} of {len(rsp_sess)} sessions")
    print("pooled regions with >= 30 units, largest rises first:")
    print(pooled.head(10).round(3).to_string())
    print(f"rank of the first RSP region: {[i for i, r in enumerate(pooled.index) if K.under(r, 'RSP', paths)][:1]} of {len(pooled)}")
print("frequency resolution (Hz):", res[names[0]]["spec_freqs"][1])

**Result: partially reproduced.** RSPv has the largest rise of the z-scored low-frequency spectral peak of all regions with at least 30 units in the minute from 5 min after the infusion (2.13), and RSPd in the minute from 10 min (2.13). RSP units rise in 7 of 10 sessions recording RSP at 5 min but in only 5 of 10 at 10 min, so the "most sessions" condition holds under one reading only.

## Summary

| Claim | Outcome |
|---|---|
| fig4-recordings-scale | reproduced |
| fig4de-response-clusters | partially reproduced |
| figs31b-cluster-overrepresentation | reproduced |
| fig4g-ketamine-alters-slow-clusters | not reproduced |
| fig4i-rise-times-unchanged | reproduced |
| fig4h-decay-shortened | reproduced |
| fig4h-decay-anatomical-progression | reproduced |
| fig5d-emotion-dimension-accumulates | partially reproduced |
| fig5e-late-reduced | not reproduced |
| fig5e-rapid-unchanged | reproduced |
| figs32c-regional-emotion-dimension | reproduced |
| fig5g-tau-persist-model | reproduced |
| figs33-alternative-models | reproduced |
| fig6d-mouse-persistent | reproduced (assumes sub-5 is the excluded animal) |
| fig6f-mouse-fast-unchanged | reproduced |
| fig6b-persistent-dimension-distributed | reproduced |
| fig6h-mouse-timescale | reproduced (assumes sub-5 is the excluded animal) |
| fig6j-mouse-coupling | partially reproduced |
| figs34d-tone-fast-unchanged | reproduced (assumes sub-5 is the excluded animal) |
| figs34e-tone-persistent | reproduced |
| figs34f-tone-shorter-than-puff | reproduced |
| figs34g-tone-timescale | partially reproduced (assumes sub-5 is the excluded animal) |
| figs34h-tone-coupling | not reproduced |
| figs30-correlation-structure | partially reproduced |
| figs29-rsp-oscillations | partially reproduced |

The claims on DANDI:001347 are in `reproduce_kauvar_2025_behavior.ipynb` or blocked; the attempt record has every outcome.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The excluded animal of Figures 5 and 6 is not named; every reading (without sub-1, sub-3 or sub-5) is run, and where they disagree sub-5 is assumed, the only reading that also gives the 7 tone sessions from 7 mice of table S3.
- The clustering gives 11 clusters, not 12, so the code's skipping of cluster 11 in the regional timing does not apply.
- Rates are binned from 0 to the last spike of the session, where brainwide-npix uses its own recording length; z-scoring is over that span.
- The tone epochs are split at the infusion time of the file (criterion) and at the code's ketamine onset, which falls before the infusion, so the code's tone Pre has 9 series, not 10.
- The driver of the paper's code is not released; the post-series window (from 1 s before the last puff) and other alignments follow the claims file's reading of the plotting code.

What was found about the dataset:

- Spike times are stored in no particular order within every one of the 9554 units; sorted, they are consistent with the stimulus table (the clock check shows a response at the table's onsets in every session and none at onsets shifted by 1.5 s).
- The file has tone series in 9 sessions from 8 mice, where table S3 reports 7 from 7, and light stimuli (240 to 248 per session in 9 sessions) that the paper does not describe.
- The infusion time in the files ("approximate") falls 94 to 420 s after the code's ketamine onset (30 s after the last puff of series 20).
- 867 units have the region label `NA`; sub-5_ses-20230706 has two probes and 154 units.
- No eye closure is in the dandiset; it is in DANDI:001347.
